# Differential Privacy Experiments: Results Comparison

This notebook loads and compares results from the DP federated learning experiments:

| Experiment | DP Type | Noise Multiplier (σ) |
|------------|---------|---------------------|
| Baseline | None | 0 |
| Local DP Low | Local | 0.5 |
| Local DP Medium | Local | 1.0 |
| Local DP High | Local | 2.0 |
| Global DP Low | Global | 0.5 |
| Global DP Medium | Global | 1.0 |
| Global DP High | Global | 2.0 |

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10

## 1. Load Results

In [ ]:
RESULTS_ROOT = Path("../results/dp_experiments")

EXPERIMENTS = {
    "baseline_no_dp": {"label": "Baseline (no DP)", "color": "black", "linestyle": "-"},
    "local_dp_low_noise": {"label": "Local DP σ=0.5", "color": "tab:blue", "linestyle": "--"},
    "local_dp_medium_noise": {"label": "Local DP σ=1.0", "color": "tab:cyan", "linestyle": "--"},
    "local_dp_high_noise": {"label": "Local DP σ=2.0", "color": "tab:purple", "linestyle": "--"},
    "global_dp_low_noise": {"label": "Global DP σ=0.5", "color": "tab:orange", "linestyle": "-."},
    "global_dp_medium_noise": {"label": "Global DP σ=1.0", "color": "tab:red", "linestyle": "-."},
    "global_dp_high_noise": {"label": "Global DP σ=2.0", "color": "tab:brown", "linestyle": "-."},
}

results = {}
for exp_name, meta in EXPERIMENTS.items():
    exp_dir = RESULTS_ROOT / exp_name
    json_path = exp_dir / "results.json"
    npz_path = exp_dir / "history.npz"

    if not json_path.exists():
        print(f"  MISSING: {exp_name}")
        continue

    with open(json_path) as f:
        data = json.load(f)

    if npz_path.exists():
        history = dict(np.load(npz_path))
        data["history_rounds"] = history["rounds"]
        data["history_losses"] = history["losses"]
        data["history_mse"] = history["mse_values"]
        data["history_rmse"] = history["rmse_values"]

    data["meta"] = meta
    results[exp_name] = data

print(f"Loaded {len(results)}/{len(EXPERIMENTS)} experiments")
for name in results:
    print(f"  ✓ {name}")

## 2. Summary Table

In [ ]:
rows = []
for exp_name, data in results.items():
    dp_info = data.get("dp", {})
    row = {
        "Experiment": data["experiment_name"],
        "DP Type": (
            "Local" if dp_info.get("local_dp")
            else "Global" if dp_info.get("global_dp")
            else "None"
        ),
        "σ (noise)": dp_info.get("noise_multiplier", 0.0),
        "Clip Norm": dp_info.get("clip_norm", "-"),
        "Final MSE": data["final_mse"],
        "Final RMSE": data["final_rmse"],
        "ε (epsilon)": dp_info.get("global_epsilon", "-"),
    }
    rows.append(row)

df_summary = pd.DataFrame(rows)
df_summary

## 3. Loss Curves Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for exp_name, data in results.items():
    meta = data["meta"]
    rounds = data.get("history_rounds", data.get("rounds", []))
    losses = data.get("history_losses", data.get("losses", []))
    if len(rounds) > 0 and len(losses) > 0:
        ax.plot(
            rounds, losses,
            label=meta["label"],
            color=meta["color"],
            linestyle=meta["linestyle"],
            linewidth=2,
            marker="o",
            markersize=4,
        )

ax.set_xlabel("Communication Round")
ax.set_ylabel("Loss")
ax.set_title("Training Loss Across Rounds", fontweight="bold")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. MSE Convergence

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for exp_name, data in results.items():
    meta = data["meta"]
    rounds = data.get("history_rounds", data.get("rounds", []))
    mse = data.get("history_mse", data.get("mse_values", []))
    if len(rounds) > 0 and len(mse) > 0:
        # Align lengths (mse may include round 0 evaluation)
        n = min(len(rounds), len(mse))
        ax.plot(
            rounds[:n], mse[:n],
            label=meta["label"],
            color=meta["color"],
            linestyle=meta["linestyle"],
            linewidth=2,
            marker="s",
            markersize=4,
        )

ax.set_xlabel("Communication Round")
ax.set_ylabel("MSE")
ax.set_title("Test MSE Across Rounds", fontweight="bold")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Final RMSE Bar Chart

In [ ]:
labels = []
rmse_vals = []
colors = []

for exp_name, data in results.items():
    meta = data["meta"]
    labels.append(meta["label"])
    rmse_vals.append(data["final_rmse"])
    colors.append(meta["color"])

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(labels)), rmse_vals, color=colors, edgecolor="k", alpha=0.8)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_ylabel("RMSE")
ax.set_title("Final Test RMSE by Experiment", fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")

# Add value labels on bars
for bar, val in zip(bars, rmse_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height(),
        f"{val:.4f}", ha="center", va="bottom", fontsize=9
    )

plt.tight_layout()
plt.show()

## 6. Privacy–Utility Trade-off

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Local DP
ax = axes[0]
local_exps = [k for k in results if "local" in k]
for exp_name in local_exps:
    data = results[exp_name]
    dp_info = data.get("dp", {})
    eps = dp_info.get("global_epsilon")
    if eps is None or eps == "-":
        continue
    meta = data["meta"]
    ax.scatter(
        eps, data["final_mse"],
        s=150, color=meta["color"], edgecolors="k", zorder=5,
    )
    ax.annotate(
        f"σ={dp_info['noise_multiplier']}",
        (eps, data["final_mse"]),
        textcoords="offset points", xytext=(8, 5), fontsize=9,
    )

# Add baseline as horizontal reference
if "baseline_no_dp" in results:
    ax.axhline(
        results["baseline_no_dp"]["final_mse"],
        color="black", linestyle=":", alpha=0.6, label="Baseline MSE",
    )

ax.set_xlabel("Privacy Budget ε (lower = more private)")
ax.set_ylabel("Final MSE (lower = better)")
ax.set_title("Local DP: Privacy–Utility Trade-off", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

# Global DP
ax = axes[1]
global_exps = [k for k in results if "global" in k]
for exp_name in global_exps:
    data = results[exp_name]
    dp_info = data.get("dp", {})
    eps = dp_info.get("global_epsilon")
    if eps is None or eps == "-":
        continue
    meta = data["meta"]
    ax.scatter(
        eps, data["final_mse"],
        s=150, color=meta["color"], edgecolors="k", zorder=5,
    )
    ax.annotate(
        f"σ={dp_info['noise_multiplier']}",
        (eps, data["final_mse"]),
        textcoords="offset points", xytext=(8, 5), fontsize=9,
    )

if "baseline_no_dp" in results:
    ax.axhline(
        results["baseline_no_dp"]["final_mse"],
        color="black", linestyle=":", alpha=0.6, label="Baseline MSE",
    )

ax.set_xlabel("Privacy Budget ε (lower = more private)")
ax.set_ylabel("Final MSE (lower = better)")
ax.set_title("Global DP: Privacy–Utility Trade-off", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Utility Degradation vs Noise Multiplier

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

baseline_rmse = results.get("baseline_no_dp", {}).get("final_rmse", 0)

# Local DP line
local_sigma = []
local_rmse = []
for exp_name in ["local_dp_low_noise", "local_dp_medium_noise", "local_dp_high_noise"]:
    if exp_name in results:
        local_sigma.append(results[exp_name]["dp"]["noise_multiplier"])
        local_rmse.append(results[exp_name]["final_rmse"])

# Global DP line
global_sigma = []
global_rmse = []
for exp_name in ["global_dp_low_noise", "global_dp_medium_noise", "global_dp_high_noise"]:
    if exp_name in results:
        global_sigma.append(results[exp_name]["dp"]["noise_multiplier"])
        global_rmse.append(results[exp_name]["final_rmse"])

if local_sigma:
    ax.plot(local_sigma, local_rmse, "o--", color="tab:blue", linewidth=2,
            markersize=8, label="Local DP")
if global_sigma:
    ax.plot(global_sigma, global_rmse, "s-.", color="tab:orange", linewidth=2,
            markersize=8, label="Global DP")

ax.axhline(baseline_rmse, color="black", linestyle=":", alpha=0.6, label="Baseline")

ax.set_xlabel("Noise Multiplier (σ)")
ax.set_ylabel("Final RMSE")
ax.set_title("Utility Degradation with Increasing Noise", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Summary

Key observations (fill in after running experiments):

- **Baseline vs DP**: ...
- **Local DP sensitivity**: ...
- **Global DP sensitivity**: ...
- **Recommended operating point**: ...